# Explore the B3 SQLite database

Use this notebook to check what was loaded by `ingest.py`.

**Before you start**
1. Run `uv sync` (once)
2. Run `uv run python ingest.py` so `data/b3.db` exists
3. Open this notebook and run the cells from top to bottom

Tip: change the SQL in the last cells to try your own queries.

## 1. Connect to the database

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd

# Path to the DB created by ingest.py
DB_PATH = Path("data") / "b3.db"

if not DB_PATH.exists():
    raise FileNotFoundError(
        f"Database not found at {DB_PATH.resolve()}.\n"
        "Run: uv run python ingest.py"
    )

conn = sqlite3.connect(DB_PATH)
print(f"Connected to: {DB_PATH.resolve()}")

Connected to: /Users/guilhermeoliveira/Documents/git_repos/b3_report/data/b3.db


## 2. List all tables

In [2]:
tables = pd.read_sql(
    """
    SELECT name AS table_name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    conn,
)
tables

,table_name
0,raw_negociacoes
1,raw_posicao_acoes
2,raw_posicao_etf
3,raw_posicao_fundos
4,raw_posicao_renda_fixa
5,raw_posicao_tesouro
6,raw_proventos


## 3. Row counts per table

In [3]:
counts = []
for table_name in tables["table_name"]:
    n = pd.read_sql(f'SELECT COUNT(*) AS n FROM "{table_name}"', conn)["n"].iloc[0]
    counts.append({"table_name": table_name, "rows": n})

pd.DataFrame(counts)

,table_name,rows
0,raw_negociacoes,1
1,raw_posicao_acoes,4
2,raw_posicao_etf,3
3,raw_posicao_fundos,6
4,raw_posicao_renda_fixa,5
5,raw_posicao_tesouro,2
6,raw_proventos,3


## 4. Peek at each table (first rows)

Change `N` if you want to see more rows.

In [4]:
N = 5

for table_name in tables["table_name"]:
    print("=" * 80)
    print(table_name)
    display(pd.read_sql(f'SELECT * FROM "{table_name}" LIMIT {N}', conn))

raw_negociacoes


,report_month,source_file,Código de Negociação,Período (Inicial),Período (Final),Instituição,Quantidade (Compra),Quantidade (Venda),Quantidade (Líquida),Preço Médio (Compra),Preço Médio (Venda)
0,2026-08,relatorio-consolidado-mensal-2026-agosto.xlsx,HGRU11,14/08/2026,-,ITAU CV S/A,10,0,10,115.5,0


raw_posicao_acoes


,report_month,source_file,Produto,Instituição,Conta,Código de Negociação,CNPJ da Empresa,Código ISIN / Distribuição,Tipo,Escriturador,Quantidade,Quantidade Disponível,Quantidade Indisponível,Motivo,Preço de Fechamento,Valor Atualizado
0,2026-08,relatorio-consolidado-mensal-2026-agosto.xlsx,AGRO3 - BRASILAGRO - CIA BRAS DE PROP AGRICOLAS,XP INVESTIMENTOS CCTVM S/A.,3112581.0,AGRO3,7.628528e+12,BRAGROACNOR7 - 116,ON,ITAU CV S/A,115.0,115.0,-,-,19.21,2209.15
1,2026-08,relatorio-consolidado-mensal-2026-agosto.xlsx,AXIA3 - AXIA ENERGIA S.A.,ITAU CV S/A,1992619.0,AXIA3,1.180000e+09,BRAXIAACNOR0 - 105,ON,ITAU CV S/A,110.0,110.0,-,-,53.12,5843.2
2,2026-08,relatorio-consolidado-mensal-2026-agosto.xlsx,AXIA3 - AXIA ENERGIA S.A.,XP INVESTIMENTOS CCTVM S/A.,3112581.0,AXIA3,1.180000e+09,BRAXIAACNOR0 - 105,ON,ITAU CV S/A,49.0,49.0,-,-,53.12,2602.88
3,2026-08,relatorio-consolidado-mensal-2026-agosto.xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,134651.81


raw_posicao_etf


...

,report_month,source_file,Produto,Instituição,Emissor,Código,Indexador,Tipo de regime,Data de Emissão,Vencimento,Quantidade,Quantidade Disponível,Quantidade Indisponível,Motivo,Contraparte,Preço Atualizado MTM,Valor Atualizado MTM,Preço Atualizado CURVA,Valor Atualizado CURVA
0,2026-08,relatorio-consolidado-mensal-2026-agosto.xlsx,CDB - BANCO C6 S.A.,BANCO C6 S.A.,BANCO C6 S.A.,CDBA2423B42,DI,REGISTRADO,03/10/2024,03/10/2030,2000.0,-,2000,-,-,-,-,1.281589,2563.17
1,2026-08,relatorio-consolidado-mensal-2026-agosto.xlsx,CDB - BANCO C6 S.A.,BANCO C6 S.A.,BANCO C6 S.A.,CDB323PZX4P,DI,REGISTRADO,28/08/2023,27/08/2029,500.0,-,500,-,-,-,-,1.440993,720.49
2,2026-08,relatorio-consolidado-mensal-2026-agosto.xlsx,CDB - BANCO C6 S.A.,BANCO C6 S.A.,BANCO C6 S.A.,CDB924A840M,DI,REGISTRADO,26/09/2024,26/09/2030,1000.0,-,1000,-,-,-,-,1.284165,1284.16
3,2026-08,relatorio-consolidado-mensal-2026-agosto.xlsx,CDB - ITAU UNIBANCO S.A.,ITAU UNIBANCO S.A.,ITAU UNIBANCO S.A.,CDB826BZHKT,DI,REGISTRADO,21/08/2026,28/07/2031,1000.0,1000,-,-,-,-,-,1.003104,1003.1
4,2026-08,relatorio-consolidado-mensal-2026-agosto.xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,NaN,243589.39


raw_posicao_tesouro


,report_month,source_file,Produto,Instituição,Código ISIN,Indexador,Vencimento,Quantidade,Quantidade Disponível,Quantidade Indisponível,Motivo,Valor Aplicado,Valor bruto,Valor líquido,Valor Atualizado
0,2026-08,relatorio-consolidado-mensal-2026-agosto.xlsx,Tesouro IPCA+ 2029,ITAU CORRETORA DE VALORES S/A,BRSTNCNTB6A3,IPCA,15/05/2029,7.49,7.49,0.0,-,24263.69,28836.03,27965.44,28836.03
1,2026-08,relatorio-consolidado-mensal-2026-agosto.xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,259621.35


raw_proventos


,report_month,source_file,Produto,Pagamento,Tipo de Evento,Instituição,Quantidade,Preço unitário,Valor líquido
0,2026-08,relatorio-consolidado-mensal-2026-agosto.xlsx,BTLG11 - BTG PACTUAL LOGISTICA FUNDO DE INVEST...,25/08/2026,Rendimento,ITAU CV S/A,125.0,0.81,101.25
1,2026-08,relatorio-consolidado-mensal-2026-agosto.xlsx,BTLG11 - BTG PACTUAL LOGISTICA FUNDO DE INVEST...,25/08/2026,Rendimento,XP INVESTIMENTOS CCTVM S/A.,75.0,0.81,60.75
2,2026-08,relatorio-consolidado-mensal-2026-agosto.xlsx,NaN,NaN,NaN,NaN,NaN,NaN,2176.76


## 5. Example queries

Edit these cells and re-run them.

In [5]:
# All stock positions for a month
pd.read_sql(
    """
    SELECT *
    FROM raw_posicao_acoes
    WHERE report_month = '2026-08'
    """,
    conn,
)

,report_month,source_file,Produto,Instituição,Conta,Código de Negociação,CNPJ da Empresa,Código ISIN / Distribuição,Tipo,Escriturador,Quantidade,Quantidade Disponível,Quantidade Indisponível,Motivo,Preço de Fechamento,Valor Atualizado
0,2026-08,relatorio-consolidado-mensal-2026-agosto.xlsx,AGRO3 - BRASILAGRO - CIA BRAS DE PROP AGRICOLAS,XP INVESTIMENTOS CCTVM S/A.,3112581.0,AGRO3,7.628528e+12,BRAGROACNOR7 - 116,ON,ITAU CV S/A,115.0,115.0,-,-,19.21,2209.15
1,2026-08,relatorio-consolidado-mensal-2026-agosto.xlsx,AXIA3 - AXIA ENERGIA S.A.,ITAU CV S/A,1992619.0,AXIA3,1.180000e+09,BRAXIAACNOR0 - 105,ON,ITAU CV S/A,110.0,110.0,-,-,53.12,5843.2
2,2026-08,relatorio-consolidado-mensal-2026-agosto.xlsx,AXIA3 - AXIA ENERGIA S.A.,XP INVESTIMENTOS CCTVM S/A.,3112581.0,AXIA3,1.180000e+09,BRAXIAACNOR0 - 105,ON,ITAU CV S/A,49.0,49.0,-,-,53.12,2602.88
3,2026-08,relatorio-consolidado-mensal-2026-agosto.xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,134651.81


In [6]:
# Provents (dividends / income) received
pd.read_sql(
    """
    SELECT *
    FROM raw_proventos
    ORDER BY Pagamento
    """,
    conn,
)

,report_month,source_file,Produto,Pagamento,Tipo de Evento,Instituição,Quantidade,Preço unitário,Valor líquido
0,2026-08,relatorio-consolidado-mensal-2026-agosto.xlsx,NaN,NaN,NaN,NaN,NaN,NaN,2176.76
1,2026-08,relatorio-consolidado-mensal-2026-agosto.xlsx,BTLG11 - BTG PACTUAL LOGISTICA FUNDO DE INVEST...,25/08/2026,Rendimento,ITAU CV S/A,125.0,0.81,101.25
2,2026-08,relatorio-consolidado-mensal-2026-agosto.xlsx,BTLG11 - BTG PACTUAL LOGISTICA FUNDO DE INVEST...,25/08/2026,Rendimento,XP INVESTIMENTOS CCTVM S/A.,75.0,0.81,60.75


In [7]:
# Negotiations / movements
pd.read_sql(
    """
    SELECT *
    FROM raw_negociacoes
    """,
    conn,
)

,report_month,source_file,Código de Negociação,Período (Inicial),Período (Final),Instituição,Quantidade (Compra),Quantidade (Venda),Quantidade (Líquida),Preço Médio (Compra),Preço Médio (Venda)
0,2026-08,relatorio-consolidado-mensal-2026-agosto.xlsx,HGRU11,14/08/2026,-,ITAU CV S/A,10,0,10,115.5,0


In [8]:
# Your own query — replace the SQL below
pd.read_sql(
    """
    SELECT report_month, COUNT(*) AS rows
    FROM raw_posicao_fundos
    GROUP BY report_month
    """,
    conn,
)

,report_month,rows
0,2026-08,6


## 6. Close the connection

Run this when you are done exploring.

In [9]:
conn.close()
print("Connection closed.")

Connection closed.
